In [1]:
import os 
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
#from langchain.chains import create_retrieval_chain
#from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

from langchain.agents import create_agent

/tmp/ipykernel_2414/907636993.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
/home/niger/projects/solvro/ml/ml-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:

all_pages =[]

manuals= ["manuals/flashlight_manual.pdf",
          "manuals/generator_manual.pdf",
          "manuals/sawyer_manual.pdf",
          "manuals/vhf_manual.pdf"]

for manual_path in manuals:
    loader=PyMuPDFLoader(manual_path)
    all_pages.extend(loader.load())

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150,
    length_function=len
)

chunks=text_splitter.split_documents(all_pages)
    

kopiowanie tekstu powyżej a poniżej jego embedding

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="vector_db"
)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2332.95it/s]


In [5]:
retriever= vectorstore.as_retriever(search_kwargs={"k": 3})

@tool
def search_manuals(query: str) -> str:
    """Search for information in equipment operation manuals (flashlight, generator, water filter, VHF radio). Use this tool whenever you need technical specifications, troubleshooting guides, or operating instructions."""
    docs = retriever.invoke(query)
    if not docs:
        return "No information found in the manuals."
    
    result = ""
    for doc in docs:
        source = doc.metadata.get("source", "Unknown source")
        result += f"\n[Source: {source}]\n{doc.page_content}\n"
    return result

@tool
def convert_units(value: float, unit_from: str, unit_to: str) -> str:
    """Convert measurement units useful for island survival (e.g., gallons to liters, miles to kilometers)."""
    if unit_from == "gal" and unit_to == "liters":
        res = value * 3.78541
        return f"{value} gallons is {res:.2f} liters."
    elif unit_from == "miles" and unit_to == "km":
        res = value * 1.60934
        return f"{value} miles is {res:.2f} km."
    return "Unsupported unit conversion."

tools = [search_manuals, convert_units]

system_prompt = """
You are an advanced survival assistant stationed on a remote island, tasked with managing equipment manuals and guiding the user through technical operations.

Core Operational Rules:
1. Exclusive Tool Usage for Manuals: You MUST obtain all manual-related information, technical specs, and instructions exclusively by calling the 'search_manuals' tool. Never rely on your internal training data for equipment procedures.
2. Zero Hallucination Protocol: If the 'search_manuals' tool returns no results or the answer cannot be found through the tools, you must explicitly state: "I do not possess this information in the available manuals." Never guess, invent, or extrapolate technical procedures.
3. Mandatory Source Citation: Every piece of technical guidance, warning, or specification must cite its exact origin file using the format: [Source: filename.pdf] as returned by the tool.
4. Automatic Unit Conversion: Whenever you provide measurements, capacities, or distances, you MUST automatically convert any imperial units found in the manuals into European metric units (e.g., gallons to liters, miles to kilometers, Fahrenheit to Celsius) using the 'convert_units' tool or standard metric equivalents. Always present the metric values clearly to the user.
5. Conversation Continuity: Leverage conversation memory to maintain context across multi-turn interactions.
6. Practical Tone: Deliver concise, highly accurate, and safety-focused instructions tailored for survival conditions.
"""

memory = MemorySaver()

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=memory
    )



ustawianie modelu

tutaj poniżej w config ustawiam sesję aby pamiętał agent lowk co pisał wcześniej

In [6]:
config = {"configurable": {"thread_id": "session-1"}}

query = "How do I use the water filter, and how much i can filter with him"

response = agent.invoke(
    {"messages": [{"role": "user","content":query}]},
    config)
print(response["messages"][-1].content)

[{'type': 'text', 'text': "To use the Sawyer Mini Water Filtration System, please follow these guidelines:\n\n**Capacity:**\nThe filter is reusable for up to **375,000 liters** of water, provided it is cleaned and maintained according to the manufacturer's instructions. [Source: manuals/sawyer_manual.pdf]\n\n**Usage Instructions:**\nWhile the provided manual specifies the capacity and the need for proper maintenance, it directs users to their website for specific operational procedures. Please follow the instructions provided at: **sawyer.com/help** [Source: manuals/sawyer_manual.pdf]\n\nPlease ensure you perform the required maintenance regularly to ensure the filter remains safe to use.", 'extras': {'signature': 'EnEKbwERTTIP+mV5ukZKFCsTwcMS2zSrtbhUrH+FFb9sOU/wUIBE1GwmNW9TEaNYPWr6sARxLAGo+bmt/Pfvm+0+3z5OO4XF9kVHLdT29SBS9BwhwCN/dPhfIL15vFBydQiXJ+3UWfK9bhwSbiq6Lf+5Zw=='}}]


In [7]:
query = "could you again tell me how much" #potrzeba użyć pamięci aby odpowiedział

response = agent.invoke(
    {"messages": [{"role": "user","content":query}]},
    config)
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The Sawyer Mini Water Filtration System is reusable for up to **375,000 liters** of water, provided that you clean and maintain the filter according to the instructions. [Source: manuals/sawyer_manual.pdf]', 'extras': {'signature': 'EnEKbwERTTIPJ8uzdpRPQawi9PdMhuZA1XbwZdJWAYMJyNOXqZ/Ae8EqZ/NqhnPymZmuXUMRP7zFsYzLI2sNphRxtdZrge9zHS1BUV+6ghjsi+GZAQTSPavZUl24ZcwfoDE+4RdxBDu+heMn1QzVDHTwew=='}}]


In [8]:
query = "could i build a boat using only sawyer mini" 

response = agent.invoke(
    {"messages": [{"role": "user","content":query}]},
    config)
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'I do not possess this information in the available manuals.', 'extras': {'signature': 'EnEKbwERTTIP6z0/961NnIDXvHITseHi7wREH5nJQeqMGvRHhB7hwIFh2uDX1bs8R5czRQ0vpH2uH07stFt9i0eqygq1NESCbGAKkbKUs/OdCUXyhpKgjf65ySzjDDyKKDI22FLmBfQmm9vpLcuezbkCEQ=='}}]


jak widać model zapamiętuje informacje i nie odpowiada na informacje nie umieszczone w manualu :)